# DADA-2000 original — Phase 2 / build the **T2** corpus  (+ Phase 3 / **Gate W**)

Plan: `.project/plans/katvad-dada-original-phase2-t2.md` · Parent: `.project/plans/katvad-dada-original-corpus.md`
Gate D0 passed on 2026-09-15 (`auc_macro` **0.6518**), which is what licenses this phase.

**What comes out:** a windowed corpus at `$KATVAD_DATA_ROOT/DADA2000_orig` (W=16, hop 8,
negatives cut from *inside* the accident videos) and one Gate-W verdict.

| Gate W | bar |
|---|---|
| length leak (clip AUC) | ≤ 0.55 |
| clip oracle (micro) | ≤ 0.75 |
| abnormal source retention | ≥ 0.90 |
| class ratio | within 1:3 either way |
| kernel coverage (median, kernel **3**) | ≤ 0.35 |
| two-class test clips | ≥ 300 |

**Any miss → STOP.** Re-open plan §4 and the parent plan's §5.2 table; do not train on a corpus that failed.

**GPU runtime.** The expensive part is I/O, not compute: `images` is **94.01 GiB** over six
spanned-zip volumes, so the extraction is **sharded** — extract → encode → delete → next.
Phase 1 already cached 400 of these clips; the loop skips them.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_CKPT_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO']      = f'{DRIVE}/kat-vad'
os.environ['DADA_ORIG'] = f"{os.environ['KATVAD_DATA_ROOT']}/DADA2000Origin/DADA2000"
os.environ['P2']        = '/content/p2'      # VM-local NVMe, never Drive
os.makedirs(os.environ['P2'], exist_ok=True)

# The T2 corpus itself is a few MB of JSON -- write it straight to Drive so a
# recycled runtime cannot take it with it (lesson C17; Phase 1 lost its census).
os.environ['T2']    = f"{os.environ['KATVAD_DATA_ROOT']}/DADA2000_orig"
# Features are keyed by SOURCE CLIP and a window is a slice, so ONE full-clip
# cache serves Phase 1 and Phase 2 alike -- and Phase 1's 400 .npy are reused.
# Never DADA2000_ncc: that is the trimmed archive (lesson C2).
os.environ['CACHE'] = f"{os.environ['KATVAD_CACHE_ROOT']}/clip/DADA2000_orig"
os.environ['EDA']   = f"{os.environ['KATVAD_OUTPUT_ROOT']}/EDA/DADA2000_orig_T2"
# The per-shard frame census is the ONE artifact that cannot be rebuilt by
# re-running anything cheap: after a shard's frames are deleted, it is the only
# record of how many images each clip had. Phase 1 kept it on /content and lost
# it to a recycled runtime; this notebook did the same thing on 2026-09-16 and
# section 3 failed with "No census file matched". It lives on Drive now.
os.environ['COUNTS'] = f"{os.environ['T2']}/counts"
os.makedirs(os.environ['COUNTS'], exist_ok=True)
print('\n'.join(f'{k:10s} {os.environ[k]}' for k in ('REPO','DADA_ORIG','T2','COUNTS','CACHE','EDA','P2')))


In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q av
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU'
df -h /content | tail -1      # section 2 sizes its shards against this

## 1. Preflight — refuse to start on a tree or an archive that is not what this notebook expects

Every assertion here failed at least once during Phases 0/1. They cost seconds; the run they
guard costs hours.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

REPO = Path(os.environ['REPO'])
ORIG = Path(os.environ['DADA_ORIG'])
P2   = Path(os.environ['P2'])
CACHE = Path(os.environ['CACHE'])

# `python -m core...` puts the CWD on sys.path, but a %%bash cd does not survive,
# and the repo is not pip-installed on Colab. PYTHONPATH is what makes it importable.
ENV = {**os.environ, 'PYTHONPATH': str(REPO)}
sys.path.insert(0, str(REPO))

# --- 1.1 the tree carries Phase 2 code at all -----------------------------
from core.data import dada_origin                      # noqa: E402
assert hasattr(dada_origin, 'census_pass'), (
    'This checkout predates Phase 2. Pull the branch that ships '
    'core/data/dada_origin.py before running anything below.')

# --- 1.2 the annotation parses to the corpus we think it is ---------------
ANNOTATION = ORIG / 'dada标注.xlsx'
assert ANNOTATION.is_file(), f'missing {ANNOTATION}'
rows = dada_origin.parse_annotation(ANNOTATION)
types = {r.type_id for r in rows}
print(f'annotation: {len(rows)} accident clips over {len(types)} types')
assert len(types) == 52, f'{len(types)} types, expected 52 -- wrong workbook?'
assert len({r.video_id for r in rows}) == len(rows), 'duplicate (type, video)'
N_CLIPS = len(rows)

# --- 1.3 every zip volume is present -------------------------------------
vols = sorted(ORIG.glob('DADA2000.z*'))
total = sum(v.stat().st_size for v in vols) / 2**30
print(f'archive: {len(vols)} volumes, {total:.1f} GiB')
for v in vols:
    print(f'   {v.name:20s} {v.stat().st_size / 2**30:6.2f} GiB')
assert len(vols) >= 2, 'a spanned archive needs its .z01.. volumes beside the .zip'

# --- 1.4 what Phase 1 already cached -------------------------------------
CACHE.mkdir(parents=True, exist_ok=True)
cached = {p.stem for p in CACHE.glob('*.npy')}
print(f'\ncache {CACHE}: {len(cached)} clips already encoded '
      f'({len(cached & {r.video_id for r in rows})} of them in this corpus)')
free = shutil.disk_usage('/content').free / 2**30
print(f'/content: {free:.1f} GiB free')

## 2.0 Archive census from the zip **index** — no extraction, no GPU

`7z l -slt` reads the archive's central directory and lists every path without
unpacking a byte. That gives an exact per-clip image count for all 1,945 clips in one pass.

Two jobs:
* **preflight** — confirms the archive holds what the annotation claims, before hours of I/O;
* **recovery** — if the per-shard censuses are ever lost (VM recycled, `/content` wiped) this
  rebuilds them from the archive, so a lost census never costs a re-extraction.

The shard censuses stay authoritative where they exist: they record what the extractor actually
read. This one records what the archive *contains*. Section 3 merges them in that order.


In [ ]:
import re   # noqa: E402

ZIP_CENSUS = Path(os.environ['COUNTS']) / 'archive_index.json'

if ZIP_CENSUS.exists():
    zip_counts = json.loads(ZIP_CENSUS.read_text())
    print(f'reusing {ZIP_CENSUS.name}: {len(zip_counts)} clips')
else:
    print('listing the archive index (no extraction; takes a minute) ...')
    proc = subprocess.run(['7z', 'l', '-slt', '-ba', 'DADA2000.zip'],
                          cwd=str(ORIG), capture_output=True, text=True)
    if proc.returncode:
        print(proc.stdout[-2000:], proc.stderr[-2000:], sep='\n')
        raise RuntimeError(f'7z l failed with exit {proc.returncode}')

    # DADA2000/{type}/{video:03d}/images/{frame}.png -- count images per clip.
    entry = re.compile(r'^DADA2000/(\d+)/(\d+)/images/[^/]+\.(png|jpg|jpeg)$', re.I)
    tally: dict[str, int] = {}
    for line in proc.stdout.splitlines():
        if not line.startswith('Path = '):
            continue
        m = entry.match(line[7:].strip().replace('\\', '/'))
        if m:
            vid = dada_origin.video_id(int(m.group(1)), int(m.group(2)))
            tally[vid] = tally.get(vid, 0) + 1
    if not tally:
        raise RuntimeError(
            '7z listed the archive but no path matched '
            'DADA2000/{type}/{video}/images/*.png -- check the layout with: '
            '!cd "$DADA_ORIG" && 7z l -ba DADA2000.zip | head -20')
    zip_counts = tally
    ZIP_CENSUS.write_text(json.dumps(zip_counts, indent=2, sort_keys=True), encoding='utf-8')
    print(f'wrote {ZIP_CENSUS} ({len(zip_counts)} clips)')

annotated = {r.video_id for r in rows}
covered = annotated & zip_counts.keys()
print(f'archive covers {len(covered)}/{len(annotated)} annotated clips')
frames_total = sum(zip_counts[v] for v in covered)
print(f'frames in those clips: {frames_total:,}  (median per clip '
      f'{sorted(zip_counts[v] for v in covered)[len(covered)//2]})')
if len(covered) < len(annotated):
    print(f'WARNING: {len(annotated) - len(covered)} annotated clips are NOT in the archive, '
          f'e.g. {sorted(annotated - zip_counts.keys())[:5]}')


## 2. Sharded extraction — extract → census + farm → encode → **delete**

`images` is 94 GiB; no runtime holds it. One shard of `SHARD` clips is ~`SHARD x 50 MiB`.

Three things this loop does that the Phase 1 loop had to learn the hard way:
* it **asserts its own input** before starting (a stale intermediate silently ran 52 clips once);
* it **skips** a shard already encoded, so a disconnect costs one shard, not the run;
* the symlink farm is built by `--census-only --flat-frames-dir`, linking at the **`images`**
  directory itself — `pathlib` will not recurse into a symlinked directory (lesson **C26**).

In [ ]:
SHARD = 150          # ~7.5 GiB of PNGs per shard; lower it if section 1.4 showed less free disk

# Drive, not /content: a census on the VM dies with the VM, and section 3 then
# cannot build the corpus even though every feature file survived (C17).
counts_dir = Path(os.environ['COUNTS'])
counts_dir.mkdir(parents=True, exist_ok=True)


def free_gib() -> float:
    return shutil.disk_usage('/content').free / 2**30


def run(cmd, cwd, capture=False):
    """Run a child and make its failure legible, not a bare CalledProcessError."""
    proc = subprocess.run([str(c) for c in cmd], cwd=str(cwd), env=ENV,
                          capture_output=capture, text=True)
    if proc.returncode:
        if capture:
            print(proc.stdout or '', proc.stderr or '', sep='\n')
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd)}'
                           + ('' if capture else '  -- scroll up for the child output'))
    if capture and proc.stdout:
        print(proc.stdout.rstrip())


assert len(rows) == N_CLIPS, 'section 1 did not run in this session'
n_shards = (len(rows) + SHARD - 1) // SHARD
print(f'{len(rows)} clips -> {n_shards} shards of up to {SHARD}\n')

for k in range(0, len(rows), SHARD):
    shard, tag = rows[k:k + SHARD], f'{k // SHARD:03d}'
    census = counts_dir / f'{tag}.json'
    if census.exists() and all((CACHE / f'{r.video_id}.npy').exists() for r in shard):
        print(f'shard {tag}: already encoded, skipping')
        continue

    frames, farm = P2 / 'frames', P2 / f'farm_{tag}'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    patterns = [f'DADA2000/{r.type_id}/{r.video:03d}/images/*' for r in shard]

    print(f'--- shard {tag}: {len(shard)} clips, {free_gib():.1f} GiB free')
    run(['7z', 'x', 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns], cwd=ORIG)
    run([sys.executable, '-m', 'core.data.dada_origin',
         '--annotation', ANNOTATION, '--frames-dir', frames,
         '--counts-out', census, '--flat-frames-dir', farm, '--census-only',
         '--out-dir', P2 / 'unused'], cwd=REPO, capture=True)
    run([sys.executable, '-m', 'core.tools.extract_clip_features',
         '--frames-dir', farm, '--dataset', 'DADA2000_orig',
         '--stride', '8', '--batch-size', '32', '--device', 'auto',
         '--no-center-crop', '--output-dir', CACHE], cwd=REPO)

    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    cache: {len(list(CACHE.glob("*.npy")))} clips, {free_gib():.1f} GiB free')

censuses = sorted(p for p in counts_dir.glob('*.json') if p.name != ZIP_CENSUS.name)
merged = {}
for c in censuses:
    merged.update(json.loads(c.read_text()))
encoded = {p.stem for p in CACHE.glob('*.npy')}
print(f'\nshards done: {len(censuses)}/{n_shards} | census: {len(merged)} clips | '
      f'cache: {len(encoded)} .npy')
missing = {r.video_id for r in rows} - encoded
if missing:
    print(f'WARNING: {len(missing)} annotated clips have no feature file, e.g. '
          f'{sorted(missing)[:5]}')


## 3. Build the T2 corpus from the merged censuses — no frames needed

The frames are gone by now; the on-disk counts survive in `counts/*.json`, which is exactly
what `--counts-file` consumes. Written straight to Drive.

In [ ]:
T2 = Path(os.environ['T2'])
counts_dir = Path(os.environ['COUNTS'])
ZIP_CENSUS = counts_dir / 'archive_index.json'

# Merge order = trust order. The archive index says what the ZIP holds; a shard
# census says what the extractor actually read. The latter wins where it exists.
census: dict[str, int] = {}
if ZIP_CENSUS.exists():
    census.update(json.loads(ZIP_CENSUS.read_text()))
shards = sorted(p for p in counts_dir.glob('*.json') if p.name != ZIP_CENSUS.name)
for c in shards:
    census.update(json.loads(c.read_text()))

if not census:
    raise SystemExit(
        f'No census under {counts_dir}.\n'
        '  - If section 2 has not run in this runtime: run section 2.0, then section 2.\n'
        '  - If the runtime was recycled: section 2.0 alone rebuilds the census from the\n'
        '    archive index without extracting anything, and the CLIP cache on Drive is\n'
        '    untouched -- check it with len(list(CACHE.glob("*.npy"))).')

# A clip with labels but no feature file fails much later, inside the EDA's
# feature section (core/eda/features.py:346). Intersect here, loudly, and hand
# dada_origin ONE resolved file instead of a glob.
encoded = {p.stem for p in CACHE.glob('*.npy')}
usable = {v: n for v, n in census.items() if v in encoded}
dropped = len(census) - len(usable)
print(f'census {len(census)} clips | encoded {len(encoded)} | usable {len(usable)}'
      + (f' | dropped {dropped} with no .npy' if dropped else ''))
if not usable:
    raise SystemExit('No clip has BOTH a census entry and a feature file; run section 2.')
if len(usable) < 0.9 * len({r.video_id for r in rows}):
    print(f'WARNING: only {len(usable)}/{len(rows)} annotated clips are usable. '
          'Section 2 is incomplete -- finish it before trusting Gate W.')

resolved = counts_dir / 'resolved_census.json'
resolved.write_text(json.dumps(usable, indent=2, sort_keys=True), encoding='utf-8')

run([sys.executable, '-m', 'core.data.dada_origin',
     '--annotation', ANNOTATION,
     '--counts-file', resolved,
     '--out-dir', T2,
     '--stride', '8',
     '--window-length', '16', '--window-stride', '8',
     '--window-max-per-clip', '4', '--window-min-positive', '1',
     '--test-ratio', '0.2', '--seed', '2024',
     '--allow-missing-frames'], cwd=REPO, capture=True)
print('\n', sorted(p.name for p in T2.iterdir()))


## 4. Corpus sanity — assert the construction before measuring it

Gate W measures the corpus; this cell checks it **is** the corpus. A build that quietly
produced 32-frame windows, or put one accident in both splits, would still produce a
plausible-looking EDA report.

In [ ]:
from core import constants   # noqa: E402

windows = json.loads((T2 / constants.WINDOWS_FILENAME).read_text())
meta    = json.loads((T2 / constants.META_FILENAME).read_text())
labels  = json.loads((T2 / constants.LABELS_TRAIN_FILENAME).read_text())
frame_test = json.loads((T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())

lengths = {w['end'] - w['start'] for w in windows.values()}
assert lengths == {16}, f'window lengths are {sorted(lengths)}, expected exactly 16'

by_split = {'train': set(), 'test': set()}
for wid, entry in meta.items():
    by_split[entry['split']].add(windows[wid]['source'])
shared = by_split['train'] & by_split['test']
assert not shared, f'{len(shared)} source clips are in BOTH splits: {sorted(shared)[:3]}'

assert set(labels.values()) == {0, 1}, (
    f'train windows carry labels {sorted(set(labels.values()))}; DVSFeatureDataset '
    'needs both classes -- T2 gets its negatives from the accident videos themselves')
assert all(e['source_is_abnormal'] for e in meta.values()), 'a normal SOURCE clip leaked in'

for key in ('texts', 'causes', 'measures', 'weather', 'light', 'scenes', 'linear'):
    assert key in next(iter(meta.values())), f'meta.json is missing {key}'

n_pos = sum(labels.values())
n_neg = len(labels) - n_pos
sources = by_split['train'] | by_split['test']
retention = len(sources) / N_CLIPS
two_class = sum(1 for v in frame_test.values() if 0 < sum(v) < len(v))

print(f'windows      : {len(windows)}  (train {len(labels)}, test {len(frame_test)})')
print(f'train classes: {n_pos} abnormal / {n_neg} normal   ratio {n_pos / max(n_neg,1):.2f}:1')
print(f'sources kept : {len(sources)}/{N_CLIPS}  retention {retention:.3f}')
print(f'two-class test windows (the auc_macro population): {two_class}')
print(f'split        : {len(by_split["train"])} train / {len(by_split["test"])} test sources')

## 5. **Gate W** — the EDA profiler on the built corpus

`--score-head-kernel 3`: T2's window is 16 sampled frames and the default kernel 9 covers
56.2 % of it (lesson **C27**). The corpus is built for kernel 3, so the coverage table must be
computed against the kernel the arms will actually train with.

In [ ]:
EDA = Path(os.environ['EDA'])
run([sys.executable, '-m', 'core.tools.eda', 'report',
     '--dataset', 'DADA2000_orig',
     '--data-dir', T2,
     '--clip-dir', CACHE,
     '--output-dir', EDA,
     '--score-head-kernel', '3',
     '--plots'], cwd=REPO)

In [ ]:
report = json.loads((EDA / constants.EDA_REPORT_JSON_FILENAME).read_text())
protocol, corpus_sec = report['protocol'], report['corpus']

leak    = protocol['clip_length_leak'].get('auc_clip_level')
oracle  = protocol['clip_constant_oracle']['auc_micro']
twoc    = protocol['macro_resolution']['two_class_clips']
cover   = corpus_sec['kernel_coverage_test']['median_coverage']
ratio   = n_pos / max(n_neg, 1)

CHECKS = [
    ('length leak (clip AUC)',      leak,      lambda v: v is not None and v <= 0.55, '<= 0.55'),
    ('clip oracle (micro AUC)',     oracle,    lambda v: v <= 0.75,                   '<= 0.75'),
    ('abnormal source retention',   retention, lambda v: v >= 0.90,                   '>= 0.90'),
    ('class ratio (abn:norm)',      ratio,     lambda v: 1/3 <= v <= 3,               'within 1:3'),
    ('kernel coverage (median, k=3)', cover,   lambda v: v <= 0.35,                   '<= 0.35'),
    ('two-class test windows',      twoc,      lambda v: v >= 300,                    '>= 300'),
]

print(f'{"criterion":34s} {"measured":>10s}  {"bar":>12s}  verdict')
print('-' * 76)
failed = []
for name, value, ok, bar in CHECKS:
    passed = ok(value)
    failed += [] if passed else [name]
    shown = 'skipped' if value is None else (f'{value:.4f}' if isinstance(value, float)
                                            else str(value))
    print(f'{name:34s} {shown:>10s}  {bar:>12s}  {"PASS" if passed else "FAIL"}')

print()
if failed:
    print(f'GATE W FAILED on: {", ".join(failed)}')
    print('STOP. Re-open plan section 4 and the parent plan section 5.2 table. Per plan '
          'risk 3, move --window-max-per-clip before --window-length: changing the window '
          'length re-fires C32 retention.')
else:
    print('GATE W PASSED -- Phase 4 (training) is unblocked.')

print('\nEDA verdicts:')
for v in report['verdicts']:
    print(f"  [{v['level']}] {v['title']} -- {v['detail']}")

## 6. Record the gate (lesson **C17**) — in the **same cell** that measured it

Phase 1's per-clip frame census existed only on a VM that was later recycled, and it is gone.
Everything this run produced is copied to Drive here, before anything else runs.

In [ ]:
import time   # noqa: E402

dest = Path(os.environ['KATVAD_OUTPUT_ROOT']) / 'EDA' / 'DADA2000_orig_T2'
dest.mkdir(parents=True, exist_ok=True)

# The census is the expensive irreplaceable artifact: it is the only record of
# how many frames each clip had before the frames were deleted.
census_dest = dest / 'counts'
census_dest.mkdir(exist_ok=True)
for c in censuses:
    shutil.copy2(c, census_dest / c.name)

proc = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
                      capture_output=True, text=True)
# `git rev-parse` printed EMPTY on the Drive mount during Phase 1 and the report
# recorded a blank commit. Say so explicitly rather than writing "".
commit = proc.stdout.strip() or f'UNKNOWN (git said: {proc.stderr.strip()!r})'

manifest = {
    'phase': 'Phase 2 (T2 corpus) + Phase 3 (Gate W)',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'commit': commit,
    'annotation': str(ANNOTATION),
    'annotated_clips': N_CLIPS,
    'data_dir': str(T2),
    'clip_cache': str(CACHE),
    'geometry': {'frame_stride': 8, 'window_length': 16, 'window_stride': 8,
                 'window_max_per_clip': 4, 'window_min_positive': 1,
                 'test_ratio': 0.2, 'seed': 2024, 'score_head_kernel': 3},
    'shards': {'size': SHARD, 'censuses': len(censuses)},
    'measured': {'windows': len(windows), 'train_windows': len(labels),
                 'test_windows': len(frame_test), 'train_abnormal': n_pos,
                 'train_normal': n_neg, 'sources_kept': len(sources),
                 'retention': retention, 'two_class_test_windows': two_class,
                 'length_leak': leak, 'clip_oracle': oracle,
                 'kernel_coverage_median': cover},
    'gate_w_failed': failed,
}
(dest / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

# meta.json is on Drive already (T2 is a Drive path) -- copied here too so the
# report folder is self-contained.
for name in (constants.META_FILENAME, constants.WINDOWS_FILENAME):
    shutil.copy2(T2 / name, dest / name)

print('recorded ->', dest)
for p in sorted(dest.rglob('*')):
    if p.is_file():
        print(f'   {p.relative_to(dest)}  ({p.stat().st_size / 1024:.1f} KiB)')
print(f'\ncommit: {commit}')
print('\nNext: plan section 6 -- Phase 4 arms, seeds 2024/2025/2026, KIP off/on, '
      'model.score_head_kernel=3, evaluated ZERO-SHOT on DoTA against the bar 0.6408. '
      'The in-domain T2 number is a sanity check only and never goes beside a published '
      'frame-level AUC (C8, C8b, C12).')